In [2]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [3]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher,CurrentYearMonthDataFetcher
from src.ghcn_daily.data_processing import WeatherDataProcessor
import numpy as np

In [5]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
data_fetcher = DataFetcher(config_file="settings.json",data_type='dataframe')
latest_data = CurrentYearMonthDataFetcher(config_file="settings.json",data_type='dataframe')

In [6]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [6]:
s_state_list=stations[stations['STATE']=='NM']['ID'].tolist()
m_state_list = stations[stations['STATE'].isin(['NM', 'TX'])]['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024)]['ID'].unique().tolist()

In [7]:
s_state_list=stations[stations['STATE']=='NH']['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024)]['ID'].unique().tolist()

In [8]:
len(s_state_list) , len(s_live_list)

(495, 151)

In [33]:
latest_df= await latest_data.save_data(s_live_list)

Fetching Data:   0%|                                                         | 0/16 [00:00<?, ?it/s]

Fetching Data: 100%|████████████████████████████████████████████████| 16/16 [00:06<00:00,  2.31it/s]


In [13]:
data= await data_fetcher.save_data(s_live_list)

Fetching Data: 100%|████████████████████████████████████████████████| 16/16 [00:08<00:00,  1.95it/s]


In [22]:
flag_columns = [col for col in data.columns if 'FLAG' in col]
data= data.drop(columns=flag_columns)
data.replace(-9999.0, np.nan, inplace=True)
weather_variables = ['TMAX', 'TMIN', 'SNOW', 'SNWD', 'PRCP']

In [35]:
processor=WeatherDataProcessor(data,weather_variables)
latest=WeatherDataProcessor(latest_df,weather_variables)

In [34]:
flag_columns = [col for col in latest_df.columns if 'FLAG' in col]
latest_df= latest_df.drop(columns=flag_columns)
latest_df.replace(-9999.0, np.nan, inplace=True)
latest_df.replace(-999.0, np.nan, inplace=True)


In [36]:
latest_l=latest.process_data()

In [ ]:
latest_l[latest_l['ID'].isin(list)]

DATE         0
ID           0
PRCP       328
TMIN      4036
SNWD      3104
TMAX      4028
SNOW      1456
Season       0
dtype: int64

In [25]:
df=processor.process_data()


In [26]:
df

,DATE,ID,PRCP,TMIN,SNWD,TMAX,SNOW,Season
0,2009-06-19,US1NHBK0001,24.9,NaN,NaN,NaN,NaN,Summer
1,2009-06-20,US1NHBK0001,4.6,NaN,NaN,NaN,NaN,Summer
2,2009-06-21,US1NHBK0001,0.0,NaN,NaN,NaN,NaN,Summer
3,2009-06-22,US1NHBK0001,1.8,NaN,NaN,NaN,NaN,Summer
4,2009-06-23,US1NHBK0001,1.0,NaN,NaN,NaN,NaN,Summer
...,...,...,...,...,...,...,...,...
1084314,2025-03-19,USW00094765,0.0,-3.8,NaN,19.4,NaN,Spring
1084315,2025-03-20,USW00094765,0.0,1.7,NaN,16.1,NaN,Spring
1084316,2025-03-21,USW00094765,11.9,-2.8,NaN,10.0,NaN,Spring
1084317,2025-03-22,USW00094765,0.3,-5.0,NaN,14.4,NaN,Spring


In [39]:
import pandas as pd
def list_stations_with_less_than_5_percent_missing(df):
    stations = df["ID"].unique()
    stations_with_less_than_5_percent_missing = []
    for station in stations:
        station_data = df[df["ID"] == station].copy()
        station_data.loc[:, 'DATE'] = pd.to_datetime(station_data['DATE'])
        station_data.sort_index(inplace=True)
        columns_to_check = ['TMIN', 'TMAX', 'SNOW', 'SNWD', 'PRCP']
        missing_percentage = station_data[columns_to_check].isnull().mean() * 100
        if (missing_percentage < 1).all():
            stations_with_less_than_5_percent_missing.append(station)
    return stations_with_less_than_5_percent_missing
list = list_stations_with_less_than_5_percent_missing(df)


In [40]:
len(list)

8

In [42]:
import pandas as pd

def count_missing_values_per_station(df, columns=['TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']):

    missing_counts = df[['ID'] + columns].groupby('ID').agg(lambda x: x.isnull().sum())
    print("Missing values per column for each station:")
    print(missing_counts)

count_missing_values_per_station(latest_l[latest_l['ID'].isin(list)])

Missing values per column for each station:
             TMIN  TMAX  PRCP  SNOW  SNWD
ID                                       
USC00272302     0     0     0     1     1
USC00272303     0     0     0     2     0
USC00275703     0     0     1     0     1
USC00275995     0     0     1     1     3
USC00276365     0     0     0     0     0
USC00278614     0     0     0     0     0
USC00279278     1     0     0     0     0
USW00014755     0     0     0     0     0


In [17]:
df[df['ID'].isin(list) & df.isnull().any(axis=1)]

,DATE,ID,PRCP,TMIN,SNWD,TMAX,SNOW,Season
309571,2008-07-20,USC00272302,3.0,NaN,0.0,NaN,0.0,Summer
309572,2008-07-21,USC00272302,508.0,NaN,0.0,NaN,0.0,Summer
309573,2008-07-22,USC00272302,249.0,NaN,0.0,NaN,0.0,Summer
309608,2008-08-26,USC00272302,0.0,NaN,0.0,289.0,0.0,Summer
310661,2011-07-15,USC00272302,0.0,122.0,NaN,261.0,NaN,Summer
...,...,...,...,...,...,...,...,...
978146,1948-05-13,USW00014755,0.0,-11.0,NaN,44.0,0.0,Spring
995859,1996-11-10,USW00014755,10.0,-106.0,NaN,-33.0,10.0,Fall
1005192,2022-05-31,USW00014755,5.0,61.0,NaN,111.0,NaN,Spring
1006091,2024-11-17,USW00014755,5.0,-33.0,NaN,0.0,3.0,Fall
